# parameter-tuning -- systematic tuning of eat-rest-v1 (baseline untouched)

Runs `external/candidates/tune_v1.py`: local random search around v1's shipped settings with successive halving.
The baseline file is never edited; every variant is the baseline's own policy class built with keyword overrides.

| Stage | Who | Seeds | Games |
|---|---|---|---|
| 1 | baseline + 80 variants (each changes 2-4 of 27 numeric settings) | 5000-5007 | 648 |
| 2 | baseline + best 12 | 6000-6023 (new) | 312 |
| 3 | baseline + best 3 | 7000-7063 (new) | 256 |

Fair comparison: every variant plays exactly the same seeds as the baseline and is scored as the paired difference
per seed. Winners are re-measured on fresh seeds at each stage, so only the FINAL table is an honest estimate. The
script ends with a verdict: `ADOPT` only if the best variant's gain exceeds twice its standard error on the 64 fresh
seeds, otherwise keep v1. Output: `logs/tune_v1/best.json`.

About 3 hours on 40 CPUs. It is launched detached, so it survives closing the laptop or the kernel, and it is
resumable: running the launch cell again continues from the finished games. Nothing else should use the CPUs meanwhile.
Mechanism study: nothing is written to `results/`.

**Cluster setup:** same as `test-eat-rest-v1.ipynb` (`.env` with `GITHUB_TOKEN=<token>`).

In [ ]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

In [ ]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

## Launch (runs in the background)

In [ ]:
import subprocess, sys
subprocess.Popen(f"mkdir -p logs && nohup {sys.executable} -u external/candidates/tune_v1.py --out logs/tune_v1 > logs/tune_v1.log 2>&1 &", shell=True)
print("started in the background -> logs/tune_v1.log")

## Progress and results (rerun any time)

In [ ]:
# Progress / results so far (rerun any time)
!grep -v "pkg_resources\|pygame\|^{" logs/tune_v1.log | tail -n 40

In [ ]:
# When finished: the verdict and the winning settings
import json, os
print(json.dumps(json.load(open("logs/tune_v1/best.json")), indent=1) if os.path.exists("logs/tune_v1/best.json") else "not finished yet")